# 🐍 Day 27 · Python Basics — Context Managers & Connecting to Several Servers (MCP + Agents)

> **Goal of this notebook:** `async with MultiServerMCPClient(...)` opens connections and closes them safely. Learn what `with` does and connect to two pretend MCP servers.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

✅ No API key and no internet needed for this notebook.

### 📋 What you will learn
1. `with` — automatic clean-up
2. Writing your own context manager (class)
3. The easy way: `@contextmanager`
4. `async with` for async connections
5. 🔗 Collect tools from two servers into one list

## 1. `with` — automatic clean-up
`with open(...)` closes the file for you, even if an error happens.

In [ ]:
with open("demo.txt", "w", encoding="utf-8") as f:
    f.write("hello")
print("Is the file closed?", f.closed)       # True — 'with' closed it

import os; os.remove("demo.txt")

## 2. Your own context manager
A class with `__enter__` (runs at the start of `with`) and `__exit__` (runs at the end — always!).

In [ ]:
class ServerConnection:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"🔌 connect to {self.name}")
        return self                     # this becomes the 'as' variable

    def __exit__(self, error_type, error, traceback):
        print(f"👋 disconnect from {self.name}")

with ServerConnection("filesystem") as conn:
    print("   working with", conn.name)

In [ ]:
# Even when an error happens inside, __exit__ still runs
try:
    with ServerConnection("database"):
        raise RuntimeError("something broke")
except RuntimeError as e:
    print("caught:", e)

## 3. The easy way: `@contextmanager`
Everything **before** `yield` = connect, everything **after** = disconnect.

In [ ]:
from contextlib import contextmanager

@contextmanager
def connection(name):
    print(f"🔌 open {name}")
    try:
        yield name
    finally:                           # 'finally' always runs
        print(f"👋 close {name}")

with connection("aws-docs") as c:
    print("   using", c)

## 4. `async with`
Network connections are async, so MCP clients use `async with` and `@asynccontextmanager`.

In [ ]:
import asyncio
from contextlib import asynccontextmanager

FAKE_SERVERS = {
    "aws-docs": ["search_documentation", "read_documentation"],
    "browser":  ["navigate", "click", "screenshot"],
}

@asynccontextmanager
async def mcp_session(server):
    await asyncio.sleep(0.2)                 # pretend the connection takes time
    print(f"🔌 connected to {server}")
    try:
        yield {"server": server, "tools": FAKE_SERVERS[server]}
    finally:
        print(f"👋 closed {server}")

async with mcp_session("browser") as session:
    print("   tools:", session["tools"])

## 5. 🔗 Tools from two servers in one list
That's what `MultiServerMCPClient` gives the agent: **one list with all tools**, each remembering which server it belongs to.

In [ ]:
from contextlib import AsyncExitStack

async def load_all_tools(servers):
    all_tools = []
    async with AsyncExitStack() as stack:            # keeps several 'async with' open together
        sessions = [await stack.enter_async_context(mcp_session(s)) for s in servers]
        for s in sessions:
            all_tools += [f"{s['server']}.{t}" for t in s["tools"]]
        print("🧰 agent can use:", all_tools)
    return all_tools

tools = await load_all_tools(["aws-docs", "browser"])
print(len(tools), "tools")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Write a `@contextmanager` called `timer()` that prints how long the code inside took.
2. Add a third fake server `"github"` with two tools and load all three.
3. What happens to the connections if an error is raised inside `load_all_tools`?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
import time
@contextmanager
def timer():
    start = time.time()
    yield
    print(f"took {time.time() - start:.2f} s")

with timer():
    sum(range(1_000_000))

FAKE_SERVERS["github"] = ["list_issues", "create_issue"]
await load_all_tools(["aws-docs", "browser", "github"])
# If an error happens, every 'finally' still runs, so all connections are closed.
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| With | `with open('a.txt') as f:` |
| Own context manager | `__enter__ / __exit__` |
| Easy version | `@contextmanager + yield` |
| Always run | `try: ... finally: ...` |
| Async version | `async with session(...) as s:` |
| Many at once | `AsyncExitStack()` |

## ➡️ What's next?
You now know the Python used in **Day_27_MCP_with_LangChain_Agents**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀